# Generador Académico RAG — LLM en Colab (GPU gratis, URL fija)

Corre **solo la generación** (el LLM) en la GPU T4 gratuita de Colab. Tu PC sigue haciendo
todo lo demás: extracción, embeddings, Chroma, retrieval y la web FastAPI. Así usás un modelo
grande (`qwen2.5:7b`/`14b`) sin GPU local, y **sin copiar ninguna URL**: el túnel usa un
dominio fijo de ngrok que ya está hardcodeado en `iniciar-colab.bat`.

### Lo único que tenés que hacer (cada demo)
1. `Entorno de ejecución → Cambiar tipo → GPU (T4)`.
2. `Entorno de ejecución → Ejecutar todo` (Ctrl+F9). Esperá ~1-2 min.
3. En tu PC: doble clic en `iniciar-colab.bat`. Listo.

### Setup por única vez
- Cuenta gratis en ngrok → copiá tu **authtoken** y reclamá tu **dominio estático** gratis
  (Dashboard → Domains, te da algo tipo `xxxx.ngrok-free.app`).
- En Colab, panel **🔑 (Secrets)** → agregá un secreto `NGROK_AUTHTOKEN` con tu token.
- Poné tu dominio en la variable `DOMAIN` de la celda 3 (acá) **y** en `iniciar-colab.bat`.

In [ ]:
# 1) Instalar Ollama (zstd: descomprime el release · pciutils: deja detectar la GPU)
!apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 2) Levantar el servidor y descargar el modelo (en GPU corre en segundos)
import os, subprocess, time

MODEL = "qwen2.5:14b-instruct"  # T4 16GB lo banca (Q4 ~9GB). Para más velocidad: qwen2.5:7b-instruct
# OLLAMA_HOST=0.0.0.0 → Ollama acepta requests con cualquier Host. Sin esto rechaza con 403
# Forbidden todo lo que llega por el túnel (su Host es el dominio ngrok, no localhost).
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.run(["pkill", "-f", "ollama"])  # matar cualquier serve previo (idempotente)
time.sleep(2)
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull {MODEL}

In [ ]:
# 3) Túnel con DOMINIO FIJO (ngrok) → la URL no cambia nunca, no hay que copiar nada
DOMAIN = "astound-cottage-smile.ngrok-free.dev"  # dominio fijo ngrok (mismo que el .bat)

!pip -q install pyngrok
from pyngrok import ngrok, conf
from google.colab import userdata

conf.get_default().auth_token = userdata.get("NGROK_AUTHTOKEN")  # secreto del panel 🔑
ngrok.kill()  # cierra túneles viejos (ngrok free = 1 a la vez)
# host_header="rewrite": ngrok reescribe el Host a localhost antes de pasar la request a Ollama.
# Sin esto Ollama responde 403 Forbidden (rechaza Hosts que no sean localhost, anti DNS-rebinding).
tunnel = ngrok.connect(11434, domain=DOMAIN, host_header="rewrite")

print("\n=== Túnel activo. NO hace falta copiar nada. ===")
print(f"URL fija : {tunnel.public_url}")
print(f"Modelo   : {MODEL}")
print("\nEn tu PC: doble clic en iniciar-colab.bat (ya tiene esta URL adentro).")
print("Mantené esta pestaña abierta durante la demo.")

In [ ]:
# 4) (opcional) Probar que responde
!curl -s https://{DOMAIN}/api/tags | head -c 300